# 02 · Load Media

**What this does:** loads the newest complete media snapshot (Prisma digital buys and Mediaocean TV buys) from Google Cloud Storage into BigQuery, so the rest of the pipeline can use it.

| | |
|---|---|
| **When to run** | Any day you want fresh media spend. New files arrive daily. |
| **What it loads** | Prisma for UP Home Ent (ZUE) and Focus (ZFF, ZFE), plus Mediaocean NetPak and SpotPak |
| **How long** | Check only: under 1 minute. Full load: about 10–20 minutes (SpotPak is large). |
| **What it changes** | Only the six raw media tables in `us_hment_mmm_raw`, and only when `DRY_RUN = False` |
| **Safe to re-run?** | Yes. A snapshot that is already loaded is skipped. |

### How it picks the date
It uses the **newest date on which every required file has arrived**, checked against each folder's delivery list (manifest). If one source is late, it uses the newest date that is complete for all of them.

### How to use it
1. Leave the settings in **Step 1** as they are and click **Run ▸ Run All Cells**. This is a *check only*; nothing changes.
2. Read the message in **Step 2**. If it says **✅ PASS — ready to load**, set `DRY_RUN = False` and **Run All** again.
3. Look at **Step 3** to confirm the snapshot is in the tables.

> Only edit the cell in Step 1. Everything below it runs on its own.

## Step 1 · Settings

In [1]:
# Which snapshot date to load: "latest" (recommended) or a specific date such as "20261030"
SNAPSHOT = "latest"

# True  = check only, nothing is changed (always start here)
# False = actually load the data
DRY_RUN = False

# Advanced: leave as False unless told otherwise
FORCE_RELOAD = False    # True = reload even if this snapshot was already loaded

## Step 2 · Run the checks and the load
*Nothing to edit here.* The detailed log prints first, followed by a short result message.

In [2]:
from IPython.display import Markdown, display

from us_hment_mmm.media.load_raw import run, show_raw_media_summary

try:
    result = run(snapshot=SNAPSHOT, dry_run=DRY_RUN, force=FORCE_RELOAD)
except Exception as exc:
    display(Markdown(
        f"### 🛑 STOP — the load failed\n\n`{exc}`\n\n"
        "Tables already finished in this run keep the new snapshot; the rest are unchanged. "
        "It is safe to fix the problem and run again. "
        "Save this notebook and send it to the pipeline owner."))
    raise

snap = result.get("snapshot")
problems = "\n".join(f"- {p}" for p in result["problems"])
messages = {
    "READY": f"### ✅ PASS — ready to load\nSnapshot **{snap}** has every required file.\n\n"
             "**Next:** in Step 1 set `DRY_RUN = False`, then **Run All** again.",
    "SUCCESS": f"### ✅ PASS — loaded\nSnapshot **{snap}** is now in BigQuery "
               f"({result.get('rows_loaded', 0):,} rows across {len(result['tables'])} tables). "
               "See Step 3.",
    "WOULD_SKIP": f"### ℹ️ Already loaded\nSnapshot **{snap}** is already in BigQuery. Nothing to do. "
                  "(To reload it anyway, set `FORCE_RELOAD = True` in Step 1.)",
    "SKIPPED": f"### ℹ️ Already loaded\nSnapshot **{snap}** is already in BigQuery. Nothing was changed.",
    "WOULD_BLOCK": f"### 🛑 STOP — no complete snapshot\n{problems}\n\n"
                   "**What to do:** files may still be arriving. Try again later today. If this "
                   "continues for more than a day, contact the pipeline owner.",
    "BLOCKED": f"### 🛑 STOP — not loaded\n{problems}\n\nNothing was changed. "
               "Contact the pipeline owner.",
}
display(Markdown(messages.get(result["status"], f"Status: {result['status']}")))

MEDIA LOAD  (LOAD)
Reading folders and manifests...

Delivery status by date (newest first):
  date        nbc-zue               nbc-zff               nbc-zfe               nbc-netpak-spotpak  
  20261030    OK                    OK                    OK                    OK                  
  20261029    OK                    OK                    OK                    MISSING             
  20261028    OK                    OK                    OK                    OK                  
  20261027    OK                    OK                    OK                    OK                  

Selected snapshot: 20261030   (14 files)
  nbc-zue              463_PrismaCampaigns_3_0_20261030102042.txt.zip                      0.0 MB (zipped)
  nbc-zue              465_PrismaPlacements_8_0_20261030055447.txt.zip                     4.7 MB (zipped)
  nbc-zue              466_PrismaAdvancedPlacementDetails_2_1_20261030061914.txt.zip       4.5 MB (zipped)
  nbc-zue              545_PrismaPlacem

### ✅ PASS — loaded
Snapshot **20261030** is now in BigQuery (6,690,935 rows across 6 tables). See Step 3.

## Step 3 · Check the result
Rows in each raw media table, by snapshot date and source folder (the newest three snapshots are kept).

In [3]:
display(show_raw_media_summary())

Table,Snapshot,Folder,Rows
mediaocean_netpak,2026-10-30,nbc-netpak-spotpak,"1,219,332"
mediaocean_spotpak,2026-10-30,nbc-netpak-spotpak,"4,179,984"
prisma_advanced_placement_details,2026-10-30,nbc-zfe,"401,911"
prisma_advanced_placement_details,2026-10-30,nbc-zff,"384,288"
prisma_advanced_placement_details,2026-10-30,nbc-zue,"404,443"
prisma_campaigns,2026-10-30,nbc-zfe,166
prisma_campaigns,2026-10-30,nbc-zff,242
prisma_campaigns,2026-10-30,nbc-zue,746
prisma_placement_buys,2026-10-30,nbc-zfe,"10,566"
prisma_placement_buys,2026-10-30,nbc-zff,"11,769"


---
### Running this as a Domino Job instead
For a long load, or to avoid keeping a workspace open, start a Domino Job with this command:

```
python -m us_hment_mmm.media.load_raw
```

Add `--dry-run` to check only.